In [1]:
import pandas as pd
from datetime import datetime
import sqlalchemy as sa
import pyodbc
import urllib
import logging

In [2]:
#in this part we are loading data from the csv file that was created today into a dataframe

date_str=datetime.now().strftime('%Y%m%d')
filename=f"export_{date_str}.csv"

df=pd.read_csv(filename,sep=';') #explicitly writing ; as separator because that's what we used in previous code

In [3]:
#adding a file for basic logging - we want to se possible errors from the DB side

logging.basicConfig(
    filename='weather_etl.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s', #timestamp, level (INFO etc.), message
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [4]:
#in this part we are connecting to our SQL Server and loading data into a staging table, before loading we want to delete all existing rows from staging table
#we could just create a new table with the same name with to_sql each time but that would possibly cause problems in the future with access rights on DB

params = urllib.parse.quote_plus("DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost\\SQLEXPRESS;DATABASE=WeatherDB;Trusted_Connection=yes;") #translating special characters into standardized URL codes

try:
    engine = sa.create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    print("connected to WeatherDB")
except Exception as e:
    print(f"error occurred: {e}")

try:
    with engine.begin() as conn: #important so that changes are commited
        conn.execute(sa.text("exec dbo.PROC_TRUNC_STG_WEATHER"))

except sa.exc.DBAPIError as db_err:
    #logging SQL Server errors
    error_msg = f"SQL Server error for PROC_TRUNC_STG_WEATHER: {db_err.orig}"
    print(error_msg)
    logging.error(error_msg)

except Exception as gen_err:
    #logging general Python errors
    error_msg = f"SQL Server error for PROC_TRUNC_STG_WEATHER: {gen_err}"
    print(error_msg)
    logging.error(error_msg)

df.to_sql("STG_WEATHER", engine, if_exists='append', index=False) #append so that the table isn't created each time data is inserted, index set to False because we don't have column index

connected to WeatherDB


C:\Users\Matej\anaconda3\Lib\contextlib.py:141: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  return next(self.gen)


23